In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install pybaseball plotly pandas numpy scipy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.1/426.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.3 MB/s eta 0:00:00


In [ ]:

# ============================================================
# CELL 2: Imports & Data Loading
# ============================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from pybaseball import batting_stats, pitching_stats, team_batting, team_pitching, playerid_lookup, statcast_batter, statcast_pitcher
from pybaseball import cache
cache.enable()

print('Loading batting stats 2024...')
batting = batting_stats(2024, qual=100)
print(f'Loaded {len(batting)} batters')

print('Loading pitching stats 2024...')
pitching = pitching_stats(2024, qual=50)
print(f'Loaded {len(pitching)} pitchers')

print('Loading team batting 2024...')
team_bat = team_batting(2024)
print(f'Loaded {len(team_bat)} teams')

print('Loading team pitching 2024...')
team_pit = team_pitching(2024)
print('All data loaded successfully!')

Loading batting stats 2024...
Loaded 455 batters
Loading pitching stats 2024...
Loaded 351 pitchers
Loading team batting 2024...
Loaded 30 teams
Loading team pitching 2024...
All data loaded successfully!


In [ ]:

# ============================================================
# CELL 3: PITCHER EVALUATION DASHBOARD
# FIP, xFIP, ERA, K%, BB%, WHIP — Top 30 Qualified Starters
# ============================================================

# Select and clean pitching columns
pitch_cols = ['Name', 'Team', 'G', 'GS', 'IP', 'ERA', 'FIP', 'xFIP', 'WHIP',
              'K/9', 'BB/9', 'K%', 'BB%', 'HR/9', 'BABIP', 'LOB%', 'WAR']
pitch_df = pitching[pitch_cols].copy()
pitch_df = pitch_df[pitch_df['GS'] >= 15].sort_values('WAR', ascending=False).head(30)

# --- CHART 1: ERA vs FIP vs xFIP Grouped Bar ---
fig1 = go.Figure()
for metric, color in zip(['ERA', 'FIP', 'xFIP'], ['#EF553B', '#636EFA', '#00CC96']):
    fig1.add_trace(go.Bar(
        name=metric,
        x=pitch_df['Name'],
        y=pitch_df[metric],
        marker_color=color,
        opacity=0.85
    ))
fig1.update_layout(
    title='<b>ERA vs FIP vs xFIP — Top 30 Starters by WAR (2024)</b>',
    barmode='group',
    xaxis_tickangle=-45,
    height=550,
    template='plotly_dark',
    legend=dict(orientation='h', y=1.1),
    yaxis_title='Rate'
)
fig1.show()

# --- CHART 2: K% vs BB% Scatter (Stuff Quality) ---
fig2 = px.scatter(
    pitch_df,
    x='BB%', y='K%',
    size='IP', color='ERA',
    hover_name='Name',
    text='Name',
    color_continuous_scale='RdYlGn_r',
    title='<b>Pitcher Stuff Quality: K% vs BB% (bubble = IP, color = ERA)</b>',
    labels={'BB%': 'Walk Rate (BB%)', 'K%': 'Strikeout Rate (K%)'},
    template='plotly_dark',
    height=600
)
fig2.update_traces(textposition='top center', textfont_size=9)
fig2.add_hline(y=pitch_df['K%'].median(), line_dash='dash', line_color='white', annotation_text='Median K%')
fig2.add_vline(x=pitch_df['BB%'].median(), line_dash='dash', line_color='white', annotation_text='Median BB%')
fig2.show()

# --- CHART 3: WAR Leaderboard ---
fig3 = px.bar(
    pitch_df.sort_values('WAR'),
    x='WAR', y='Name',
    orientation='h',
    color='WAR',
    color_continuous_scale='Blues',
    title='<b>Pitcher WAR Leaderboard — Top 30 Starters (2024)</b>',
    template='plotly_dark',
    height=700
)
fig3.update_layout(yaxis={'categoryorder': 'total ascending'})
fig3.show()

print('Pitcher Dashboard complete.')

Pitcher Dashboard complete.


In [ ]:

# ============================================================
# CELL 4: HITTER EVALUATION DASHBOARD
# wRC+, OBP, SLG, wOBA, ISO, BABIP, BB%, K%
# ============================================================

bat_cols = ['Name', 'Team', 'G', 'PA', 'HR', 'R', 'RBI', 'SB',
            'AVG', 'OBP', 'SLG', 'OPS', 'wOBA', 'wRC+',
            'BB%', 'K%', 'ISO', 'BABIP', 'WAR']
bat_df = batting[bat_cols].copy()
bat_df = bat_df[bat_df['PA'] >= 300].sort_values('WAR', ascending=False).head(40)

# --- CHART 1: wRC+ Leaderboard ---
fig4 = px.bar(
    bat_df.sort_values('wRC+'),
    x='wRC+', y='Name',
    orientation='h',
    color='wRC+',
    color_continuous_scale='Viridis',
    title='<b>Hitter wRC+ Leaderboard — Top 40 by WAR (2024)</b>',
    template='plotly_dark',
    height=800
)
fig4.add_vline(x=100, line_dash='dash', line_color='yellow', annotation_text='League Avg (100)')
fig4.update_layout(yaxis={'categoryorder': 'total ascending'})
fig4.show()

# --- CHART 2: OBP vs SLG Scatter (4-Quadrant) ---
fig5 = px.scatter(
    bat_df,
    x='OBP', y='SLG',
    size='WAR', color='wRC+',
    hover_name='Name',
    text='Name',
    color_continuous_scale='RdYlGn',
    title='<b>OBP vs SLG — 4-Quadrant Hitter Profile (bubble = WAR)</b>',
    template='plotly_dark',
    height=650
)
fig5.update_traces(textposition='top center', textfont_size=8)
fig5.add_hline(y=bat_df['SLG'].median(), line_dash='dash', line_color='gray')
fig5.add_vline(x=bat_df['OBP'].median(), line_dash='dash', line_color='gray')
fig5.show()

# --- CHART 3: Plate Discipline — BB% vs K% ---
fig6 = px.scatter(
    bat_df,
    x='K%', y='BB%',
    size='PA', color='wOBA',
    hover_name='Name',
    text='Name',
    color_continuous_scale='RdYlGn',
    title='<b>Plate Discipline: BB% vs K% (bubble = PA, color = wOBA)</b>',
    template='plotly_dark',
    height=600
)
fig6.update_traces(textposition='top center', textfont_size=8)
fig6.add_hline(y=bat_df['BB%'].median(), line_dash='dash', line_color='white', annotation_text='Median BB%')
fig6.add_vline(x=bat_df['K%'].median(), line_dash='dash', line_color='white', annotation_text='Median K%')
fig6.show()

# --- CHART 4: Triple Slash Table ---
fig7 = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Name</b>', '<b>Team</b>', '<b>AVG</b>', '<b>OBP</b>',
                '<b>SLG</b>', '<b>OPS</b>', '<b>wOBA</b>', '<b>wRC+</b>',
                '<b>HR</b>', '<b>RBI</b>', '<b>WAR</b>'],
        fill_color='#1f2c56',
        font=dict(color='white', size=12),
        align='center'
    ),
    cells=dict(
        values=[
            bat_df['Name'], bat_df['Team'],
            bat_df['AVG'].round(3), bat_df['OBP'].round(3),
            bat_df['SLG'].round(3), bat_df['OPS'].round(3),
            bat_df['wOBA'].round(3), bat_df['wRC+'].round(1),
            bat_df['HR'], bat_df['RBI'], bat_df['WAR'].round(1)
        ],
        fill_color=[['#0d1b2a' if i%2==0 else '#162032' for i in range(len(bat_df))]]*11,
        font=dict(color='white', size=11),
        align='center'
    )
)])
fig7.update_layout(
    title='<b>2024 Hitter Sabermetrics Table — Top 40 by WAR</b>',
    template='plotly_dark',
    height=700
)
fig7.show()

print('Hitter Dashboard complete.')

Hitter Dashboard complete.


In [ ]:

# ============================================================
# CELL 5: TEAM-LEVEL ANALYTICS DASHBOARD
# Offense: Runs, wRC+, OPS, HR | Defense: ERA, WHIP, FIP
# ============================================================

# --- Team Batting ---
tbat_cols = ['teamIDfg', 'G', 'PA', 'HR', 'R', 'RBI', 'AVG', 'OBP', 'SLG', 'OPS', 'wOBA', 'wRC+']
tbat = team_bat[tbat_cols].copy().rename(columns={'teamIDfg': 'Team'})

# --- Team Pitching ---
tpit_cols = ['teamIDfg', 'ERA', 'FIP', 'xFIP', 'WHIP', 'K/9', 'BB/9', 'HR/9', 'WAR']
tpit = team_pit[tpit_cols].copy().rename(columns={'teamIDfg': 'Team'})

# Merge
team_df = pd.merge(tbat, tpit, on='Team')

# --- CHART 1: Runs Scored Leaderboard ---
fig8 = px.bar(
    team_df.sort_values('R'),
    x='R', y='Team',
    orientation='h',
    color='wRC+',
    color_continuous_scale='RdYlGn',
    title='<b>2024 Team Runs Scored & wRC+ (color)</b>',
    template='plotly_dark',
    height=700
)
fig8.update_layout(yaxis={'categoryorder': 'total ascending'})
fig8.show()

# --- CHART 2: Offense vs Defense Scatter ---
fig9 = px.scatter(
    team_df,
    x='ERA', y='OPS',
    size='WAR',
    color='R',
    hover_name='Team',
    text='Team',
    color_continuous_scale='Blues',
    title='<b>Team Offense (OPS) vs Defense (ERA) — 2024</b>',
    labels={'ERA': 'Team ERA (lower=better)', 'OPS': 'Team OPS (higher=better)'},
    template='plotly_dark',
    height=600
)
fig9.update_traces(textposition='top center', textfont_size=10)
fig9.add_hline(y=team_df['OPS'].median(), line_dash='dash', line_color='white', annotation_text='Median OPS')
fig9.add_vline(x=team_df['ERA'].median(), line_dash='dash', line_color='white', annotation_text='Median ERA')
fig9.show()

# --- CHART 3: Team FIP vs ERA (pitching luck) ---
fig10 = go.Figure()
fig10.add_trace(go.Bar(name='ERA', x=team_df['Team'], y=team_df['ERA'], marker_color='#EF553B', opacity=0.8))
fig10.add_trace(go.Bar(name='FIP', x=team_df['Team'], y=team_df['FIP'], marker_color='#636EFA', opacity=0.8))
fig10.add_trace(go.Bar(name='xFIP', x=team_df['Team'], y=team_df['xFIP'], marker_color='#00CC96', opacity=0.8))
fig10.update_layout(
    title='<b>Team ERA vs FIP vs xFIP — Pitching Luck Indicator (2024)</b>',
    barmode='group',
    xaxis_tickangle=-45,
    template='plotly_dark',
    height=550,
    yaxis_title='Rate',
    legend=dict(orientation='h', y=1.05)
)
fig10.show()

# --- CHART 4: Full Team Summary Table ---
fig11 = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Team</b>', '<b>R</b>', '<b>HR</b>', '<b>OPS</b>',
                '<b>wRC+</b>', '<b>ERA</b>', '<b>FIP</b>', '<b>WHIP</b>', '<b>P-WAR</b>'],
        fill_color='#1a1a2e',
        font=dict(color='white', size=12),
        align='center'
    ),
    cells=dict(
        values=[
            team_df['Team'], team_df['R'], team_df['HR'],
            team_df['OPS'].round(3), team_df['wRC+'].round(1),
            team_df['ERA'].round(2), team_df['FIP'].round(2),
            team_df['WHIP'].round(3), team_df['WAR'].round(1)
        ],
        fill_color=[['#0d1117' if i%2==0 else '#161b22' for i in range(len(team_df))]]*9,
        font=dict(color='white', size=11),
        align='center'
    )
)])
fig11.update_layout(
    title='<b>2024 MLB Full Team Analytics Summary</b>',
    template='plotly_dark',
    height=750
)
fig11.show()

print('Team Dashboard complete.')

Team Dashboard complete.


In [ ]:

# ============================================================
# CELL 6: WAR COMPARISON DASHBOARD + EXPORT
# Batting vs Pitching WAR, Top 10 each, and CSV export
# ============================================================

# Top hitters by WAR
top_batters_war = bat_df.nlargest(15, 'WAR')[['Name', 'Team', 'WAR', 'wRC+', 'OPS']].copy()
top_batters_war['Type'] = 'Hitter'

# Top pitchers by WAR
top_pitchers_war = pitch_df.nlargest(15, 'WAR')[['Name', 'Team', 'WAR', 'ERA', 'FIP']].copy()
top_pitchers_war['Type'] = 'Pitcher'

# --- CHART 1: Side-by-side WAR comparison ---
fig12 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Top 15 Hitters by WAR</b>', '<b>Top 15 Pitchers by WAR</b>')
)

fig12.add_trace(go.Bar(
    x=top_batters_war['WAR'],
    y=top_batters_war['Name'],
    orientation='h',
    marker_color='#00CC96',
    name='Hitters'
), row=1, col=1)

fig12.add_trace(go.Bar(
    x=top_pitchers_war['WAR'],
    y=top_pitchers_war['Name'],
    orientation='h',
    marker_color='#636EFA',
    name='Pitchers'
), row=1, col=2)

fig12.update_layout(
    title='<b>2024 MLB WAR Leaders — Hitters vs Pitchers</b>',
    template='plotly_dark',
    height=600,
    showlegend=True
)
fig12.update_yaxes(categoryorder='total ascending', row=1, col=1)
fig12.update_yaxes(categoryorder='total ascending', row=1, col=2)
fig12.show()

# --- CHART 2: Combined WAR + Metric Scatter ---
combined = pd.concat([
    top_batters_war[['Name', 'Team', 'WAR', 'Type']],
    top_pitchers_war[['Name', 'Team', 'WAR', 'Type']]
])

fig13 = px.bar(
    combined.sort_values('WAR'),
    x='WAR', y='Name',
    color='Type',
    orientation='h',
    color_discrete_map={'Hitter': '#00CC96', 'Pitcher': '#636EFA'},
    title='<b>Top 30 MLB Players by WAR (Hitters + Pitchers) — 2024</b>',
    template='plotly_dark',
    height=750
)
fig13.update_layout(yaxis={'categoryorder': 'total ascending'})
fig13.show()

# ============================================================
# EXPORT ALL DATAFRAMES TO CSV
# ============================================================
bat_df.to_csv('2024_hitter_sabermetrics.csv', index=False)
pitch_df.to_csv('2024_pitcher_sabermetrics.csv', index=False)
team_df.to_csv('2024_team_analytics.csv', index=False)

print('\n=== BASEBALL ANALYTICS DASHBOARD COMPLETE ===')
print('Files exported:')
print('  2024_hitter_sabermetrics.csv')
print('  2024_pitcher_sabermetrics.csv')
print('  2024_team_analytics.csv')
print('\nDashboard includes:')
print('  [1] ERA vs FIP vs xFIP pitcher comparison')
print('  [2] Pitcher K% vs BB% stuff quality scatter')
print('  [3] Pitcher WAR leaderboard')
print('  [4] Hitter wRC+ leaderboard')
print('  [5] OBP vs SLG 4-quadrant scatter')
print('  [6] Plate discipline BB% vs K%')
print('  [7] Hitter sabermetrics table')
print('  [8] Team runs scored & wRC+')
print('  [9] Team offense vs defense scatter')
print(' [10] Team ERA vs FIP vs xFIP')
print(' [11] Full team analytics table')
print(' [12] WAR leaders: Hitters vs Pitchers')
print(' [13] Combined top 30 WAR leaderboard')


=== BASEBALL ANALYTICS DASHBOARD COMPLETE ===
Files exported:
  2024_hitter_sabermetrics.csv
  2024_pitcher_sabermetrics.csv
  2024_team_analytics.csv

Dashboard includes:
  [1] ERA vs FIP vs xFIP pitcher comparison
  [2] Pitcher K% vs BB% stuff quality scatter
  [3] Pitcher WAR leaderboard
  [4] Hitter wRC+ leaderboard
  [5] OBP vs SLG 4-quadrant scatter
  [6] Plate discipline BB% vs K%
  [7] Hitter sabermetrics table
  [8] Team runs scored & wRC+
  [9] Team offense vs defense scatter
 [10] Team ERA vs FIP vs xFIP
 [11] Full team analytics table
 [12] WAR leaders: Hitters vs Pitchers
 [13] Combined top 30 WAR leaderboard
